### **Sieć neuronowa MTL na reprezentacji fingerprint - zestaw 1 - Absorpcja**

Wykorzystana reprezentacja: **ECFP4**

Lista endpointów:


1. Caco-2 (Wang)
2. Lipophilicity (AstraZeneca)
3. Solubility (AqSolDB)
4. HIA (Hou)
5. AMES Mutagenicity

Wyniki dla STL:

In [43]:
!pip install rdkit
!pip install pandas numpy scikit-learn -U
!pip install pytdc --no-dependencies
!pip install fuzzywuzzy

In [44]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [45]:
data_folder = "/content/drive/MyDrive/mldd_data" #folder z zapisanymi zestawami danych aby umożliwić porównanie danych na dokładnie tych samych zestawach dla każdego modelu

#data_folder = "/content/drive/MyDrive/MLDD - ADMET/data_splits"

In [46]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from tdc.single_pred import ADME
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, roc_auc_score, accuracy_score, f1_score

In [47]:
class Featurizer:
    def __init__(self, y_column, smiles_col='Drug', **kwargs):
        self.y_column = y_column
        self.smiles_col = smiles_col
        self.__dict__.update(kwargs)
    def __call__(self, df):
        raise NotImplementedError()

class ECFPFeaturizer(Featurizer): #dodane zabezpieczenia na Nan etc
    def __init__(self, y_column='Y', radius=2, length=1024, **kwargs):
        self.radius = radius
        self.length = length
        super().__init__(y_column, **kwargs)

    def __call__(self, df):
        fingerprints = []
        labels = []
        for i, row in df.iterrows():
            smiles = row[self.smiles_col]
            mol = Chem.MolFromSmiles(str(smiles)) if pd.notna(smiles) else None

            if mol:
                fp = AllChem.GetMorganFingerprintAsBitVect(mol, self.radius, nBits=self.length)
                fingerprints.append(np.array(fp))
            else:
                # Zamiast pomijać wiersz, wstawiamy wektor zer (zachowuje wyrównanie)
                fingerprints.append(np.zeros(self.length))

            # Pobieramy etykietę jeśli istnieje (dla dummy X wstawiamy NaN)
            labels.append(row[self.y_column] if self.y_column in df.columns else np.nan)

        return np.array(fingerprints), np.array(labels).reshape(-1, 1)

In [48]:

# DEFINICJA SIECI NEURONOWEJ

class AdmetEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, dropout=0.2):
        super().__init__()
        self.main = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.res_layer = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        x = self.main(x)
        return torch.relu(x + self.res_layer(x))


class MTL_NN_Hybrid(nn.Module):
    def __init__(self, input_dim, reg_tasks, class_tasks, hidden_dim=512):
        """
        reg_tasks: lista nazw zadań regresyjnych (np. ['Lipophilicity', 'Solubility'])
        class_tasks: lista nazw zadań klasyfikacyjnych (np. ['BBB', 'Ames'])
        """
        super().__init__()
        self.encoder = AdmetEncoder(input_dim, hidden_dim)
        self.reg_tasks = reg_tasks
        self.class_tasks = class_tasks

        # Słowniki głowic dla każdego typu zadania
        #Pozwala zapisać wiele warstw Linear i nazwać je tak, jak nazywają się Twoje zadania.
        self.reg_heads = nn.ModuleDict({
            task: nn.Linear(hidden_dim, 1) for task in reg_tasks
        })
        self.class_heads = nn.ModuleDict({
            task: nn.Linear(hidden_dim, 1) for task in class_tasks
        })

    def forward(self, x):
        shared_features = self.encoder(x) #wspólny enkoder
        results = {}

        # Procesowanie zadań regresyjnych
        for task in self.reg_tasks:
            results[task] = self.reg_heads[task](shared_features)

        # Procesowanie zadań klasyfikacyjnych
        for task in self.class_tasks:
            # results[task] = torch.sigmoid(self.class_heads[task](shared_features))
            results[task] = self.class_heads[task](shared_features)

        #print(results)
        return results #Zwracając słownik:{'Caco2_Wang': 1.2, 'Lipophilicity_AZ': 0.5} masz absolutną pewność, który wynik dotyczy którego badania.

In [49]:

# --- STL REGRESOR ---
class STL_NN_Regressor(nn.Module):
    def __init__(self, input_dim, hidden_dim=512):
        super().__init__()
        self.encoder = AdmetEncoder(input_dim, hidden_dim)
        self.head = nn.Linear(hidden_dim, 1) # Wynik liniowy (dowolna liczba)

    def forward(self, x):
        return self.head(self.encoder(x))

# --- STL KLASYFIKATOR ---
class STL_NN_Classifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=512):
        super().__init__()
        self.encoder = AdmetEncoder(input_dim, hidden_dim)
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # Sigmoid zamienia wynik na prawdopodobieństwo (0-1)
        return torch.sigmoid(self.head(self.encoder(x)))

  #=============================

In [50]:
def train_MTL_hybrid_wl_sum(X_train_np, X_test_np, y_train_dict, y_test_dict, reg_tasks, class_tasks):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # --- 1. Normalizacja (osobny skaler dla każdego zadania regresyjnego) ---
    scalers = {}
    y_train_tensors = {}
    y_test_tensors = {}

    for task in reg_tasks:
        scalers[task] = StandardScaler()
        # Wyciągamy dane i maskujemy NaN, aby skaler zadziałał
        train_vals = y_train_dict[task].reshape(-1, 1)
        mask = ~np.isnan(train_vals).flatten()

        y_train_scaled = np.full_like(train_vals, np.nan)
        if mask.any():
            y_train_scaled[mask] = scalers[task].fit_transform(train_vals[mask])

        y_train_tensors[task] = torch.FloatTensor(y_train_scaled).to(device)
        # Testowe przesyłamy w oryginale (skalujemy tylko przy ewaluacji dla wygody)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    for task in class_tasks:
        # Klasyfikacja nie wymaga skalowania
        y_train_tensors[task] = torch.FloatTensor(y_train_dict[task].reshape(-1, 1)).to(device)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    X_train_t = torch.FloatTensor(X_train_np).to(device)
    X_test_t = torch.FloatTensor(X_test_np).to(device)

    # --- 2. Model i Optymalizator ---
    model = MTL_NN_Hybrid(input_dim=1024, reg_tasks=reg_tasks, class_tasks=class_tasks).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion_reg = nn.MSELoss(reduction='none') # 'none' pozwala nam ręcznie obsłużyć NaN
    criterion_cls = nn.BCEWithLogitsLoss(reduction='none')

    print("\n--- START TRENINGU (Multi-Task) ---")
    model.train()
    for epoch in range(100):
        optimizer.zero_grad()
        outputs = model(X_train_t)
        total_loss = 0

        # Sumujemy straty ze wszystkich zadań, ignorując NaN
        for task in reg_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_reg(outputs[task][mask], target[mask]).mean()
                total_loss += loss

        for task in class_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_cls(outputs[task][mask], target[mask]).mean()
                total_loss += loss

        total_loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            print(f"  Epoka {epoch:3d} | Total Loss: {total_loss.item():.4f}")

    # --- 3. Ewaluacja ---
    print("\n--- EWALUACJA ---")
    model.eval()
    all_metrics = {"reg_tasks": {}, "class_tasks": {}}

    with torch.no_grad():
        preds_dict = model(X_test_t)

        for task in reg_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            # Odwracamy skalowanie
            p_scaled = preds_dict[task].cpu().numpy()
            p_unscaled = scalers[task].inverse_transform(p_scaled).flatten()


            mse = mean_squared_error(y_true[mask], p_unscaled[mask])
            mae = mean_absolute_error(y_true[mask], p_unscaled[mask]) # <-- Dodaj tę linię
            r2 = r2_score(y_true[mask], p_unscaled[mask])

            all_metrics["reg_tasks"][task] = {
               "rmse": np.sqrt(mse),
               "mae": mae,  # <-- I tę linię
               "r2": r2
             }

        for task in class_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            p_probs  = torch.sigmoid(preds_dict[task]).cpu().numpy().flatten()
            p_labels = (p_probs[mask] > 0.5).astype(int)
            try:
                auroc = roc_auc_score(y_true[mask], p_probs[mask])
            except ValueError:
                auroc = 0.5

            all_metrics["class_tasks"][task] = {
                "accuracy": accuracy_score(y_true[mask], p_labels),
                "f1":       f1_score(y_true[mask], p_labels),
                "auroc":    auroc,
            }

    return all_metrics

In [51]:
def train_MTL_hybrid_uniform_average(X_train_np, X_test_np, y_train_dict, y_test_dict, reg_tasks, class_tasks):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    scalers = {}
    y_train_tensors = {}
    y_test_tensors = {}

    for task in reg_tasks:
        scalers[task] = StandardScaler()
        train_vals = y_train_dict[task].reshape(-1, 1)
        mask = ~np.isnan(train_vals).flatten()

        y_train_scaled = np.full_like(train_vals, np.nan)
        if mask.any():
            y_train_scaled[mask] = scalers[task].fit_transform(train_vals[mask])

        y_train_tensors[task] = torch.FloatTensor(y_train_scaled).to(device)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    for task in class_tasks:
        y_train_tensors[task] = torch.FloatTensor(y_train_dict[task].reshape(-1, 1)).to(device)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    X_train_t = torch.FloatTensor(X_train_np).to(device)
    X_test_t = torch.FloatTensor(X_test_np).to(device)

    model = MTL_NN_Hybrid(input_dim=1024, reg_tasks=reg_tasks, class_tasks=class_tasks).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion_reg = nn.MSELoss(reduction='none')
    criterion_cls = nn.BCEWithLogitsLoss(reduction='none')

    print("\n--- START TRENINGU (Multi-Task - Uniform Average) ---")
    model.train()
    for epoch in range(100):
        optimizer.zero_grad()
        outputs = model(X_train_t)

        task_losses = []

        for task in reg_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_reg(outputs[task][mask], target[mask]).mean()
                task_losses.append(loss)

        for task in class_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_cls(outputs[task][mask], target[mask]).mean()
                task_losses.append(loss)

        if task_losses:
            total_loss = torch.stack(task_losses).mean() # Uniform average
        else:
            total_loss = torch.tensor(0.0, device=device, requires_grad=True)

        total_loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            print(f"  Epoka {epoch:3d} | Total Loss: {total_loss.item():.4f}")

    print("\n--- EWALUACJA ---")
    model.eval()
    all_metrics = {"reg_tasks": {}, "class_tasks": {}}

    with torch.no_grad():
        preds_dict = model(X_test_t)

        for task in reg_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            p_scaled = preds_dict[task].cpu().numpy()
            p_unscaled = scalers[task].inverse_transform(p_scaled).flatten()

            mse = mean_squared_error(y_true[mask], p_unscaled[mask])
            mae = mean_absolute_error(y_true[mask], p_unscaled[mask])
            r2 = r2_score(y_true[mask], p_unscaled[mask])

            all_metrics["reg_tasks"][task] = {
               "rmse": np.sqrt(mse),
               "mae": mae,
               "r2": r2
             }

        for task in class_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            p_probs  = torch.sigmoid(preds_dict[task]).cpu().numpy().flatten()
            p_labels = (p_probs[mask] > 0.5).astype(int)
            try:
                auroc = roc_auc_score(y_true[mask], p_probs[mask])
            except ValueError:
                auroc = 0.5

            all_metrics["class_tasks"][task] = {
                "accuracy": accuracy_score(y_true[mask], p_labels),
                "f1":       f1_score(y_true[mask], p_labels),
                "auroc":    auroc,
            }

    return all_metrics

In [52]:
def train_MTL_hybrid_uncertainty_weighting(X_train_np, X_test_np, y_train_dict, y_test_dict, reg_tasks, class_tasks):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    scalers = {}
    y_train_tensors = {}
    y_test_tensors = {}

    for task in reg_tasks:
        scalers[task] = StandardScaler()
        train_vals = y_train_dict[task].reshape(-1, 1)
        mask = ~np.isnan(train_vals).flatten()

        y_train_scaled = np.full_like(train_vals, np.nan)
        if mask.any():
            y_train_scaled[mask] = scalers[task].fit_transform(train_vals[mask])

        y_train_tensors[task] = torch.FloatTensor(y_train_scaled).to(device)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    for task in class_tasks:
        y_train_tensors[task] = torch.FloatTensor(y_train_dict[task].reshape(-1, 1)).to(device)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    X_train_t = torch.FloatTensor(X_train_np).to(device)
    X_test_t = torch.FloatTensor(X_test_np).to(device)

    model = MTL_NN_Hybrid(input_dim=1024, reg_tasks=reg_tasks, class_tasks=class_tasks).to(device)

    # Learnable parameters for task uncertainty
    log_vars = nn.ParameterDict()
    for task in reg_tasks + class_tasks:
        log_vars[task] = nn.Parameter(torch.zeros(1, device=device)) # Initialize log_sigma_sq to 0

    # Include log_vars in the optimizer
    optimizer = optim.Adam(list(model.parameters()) + list(log_vars.values()), lr=0.001)
    criterion_reg = nn.MSELoss(reduction='none')
    criterion_cls = nn.BCEWithLogitsLoss(reduction='none')

    print("\n--- START TRENINGU (Multi-Task - Uncertainty Weighting) ---")
    model.train()
    for epoch in range(100):
        optimizer.zero_grad()
        outputs = model(X_train_t)
        total_loss = 0

        for task in reg_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_reg(outputs[task][mask], target[mask])
                precision = torch.exp(-log_vars[task])
                total_loss += torch.mean(0.5 * precision * loss + 0.5 * log_vars[task])

        for task in class_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_cls(outputs[task][mask], target[mask])
                precision = torch.exp(-log_vars[task])
                total_loss += torch.mean(precision * loss + 0.5 * log_vars[task])

        total_loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            print(f"  Epoka {epoch:3d} | Total Loss: {total_loss.item():.4f}")
            # print(f"    Log_vars: {{ {', '.join([f'{t}: {log_vars[t].item():.2f}' for t in log_vars])} }}")

    print("\n--- EWALUACJA ---")
    model.eval()
    all_metrics = {"reg_tasks": {}, "class_tasks": {}}

    with torch.no_grad():
        preds_dict = model(X_test_t)

        for task in reg_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            p_scaled = preds_dict[task].cpu().numpy()
            p_unscaled = scalers[task].inverse_transform(p_scaled).flatten()

            mse = mean_squared_error(y_true[mask], p_unscaled[mask])
            mae = mean_absolute_error(y_true[mask], p_unscaled[mask])
            r2 = r2_score(y_true[mask], p_unscaled[mask])

            all_metrics["reg_tasks"][task] = {
               "rmse": np.sqrt(mse),
               "mae": mae,
               "r2": r2
             }

        for task in class_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            p_probs  = torch.sigmoid(preds_dict[task]).cpu().numpy().flatten()
            p_labels = (p_probs[mask] > 0.5).astype(int)
            try:
                auroc = roc_auc_score(y_true[mask], p_probs[mask])
            except ValueError:
                auroc = 0.5

            all_metrics["class_tasks"][task] = {
                "accuracy": accuracy_score(y_true[mask], p_labels),
                "f1":       f1_score(y_true[mask], p_labels),
                "auroc":    auroc,
            }

    return all_metrics

In [53]:
import pickle

def load_split_pickle(dataset_name):
    filepath = f"{data_folder}/{dataset_name}_split.pkl"

    with open(filepath, "rb") as f:
        split = pickle.load(f)

    return split["train"], split["test"]

In [54]:
def print_metrics(metrics, task='classification', weight_loss_func_name=None):
    print(f"\n{'='*40}")
    if weight_loss_func_name: # Check if func_name is provided and not None
        print(f"  Loss Weighting: {weight_loss_func_name}")
        print(f"{'='*40}")
    if task == 'classification':
        print(f"  Accuracy : {metrics['test_metrics']['accuracy']:.4f}")
        print(f"  F1       : {metrics['test_metrics']['f1']:.4f}")
        print(f"  AUROC    : {metrics['test_metrics']['auroc']:.4f}")
    else:
        print(f"  RMSE     : {metrics['test_metrics']['rmse']:.4f}")
        print(f"  MAE      : {metrics['test_metrics']['mae']:.4f}")
        print(f"  R²       : {metrics['test_metrics']['r2']:.4f}")
    print(f"{'='*40}\n")


def save_metrics(metrics, dataset_name, filepath, task='classification', weight_loss_func_name=None, endpoint_group_name=None):
    with open(filepath, 'a') as f:
        f.write(f"\n{'='*40}\n")
        f.write(f"Endpoint    : {dataset_name}\n")
        if endpoint_group_name:
            f.write(f"Tasks       : {endpoint_group_name}\n")
        if weight_loss_func_name:
            f.write(f"Loss Weighting: {weight_loss_func_name}\n")
        f.write(f"{'='*40}\n")
        if task == 'classification':
            f.write(f"  Accuracy : {metrics['test_metrics']['accuracy']:.4f}\n")
            f.write(f"  F1       : {metrics['test_metrics']['f1']:.4f}\n")
            f.write(f"  AUROC    : {metrics['test_metrics']['auroc']:.4f}\n")
        else:
            f.write(f"  RMSE     : {metrics['test_metrics']['rmse']:.4f}\n")
            f.write(f"  MAE      : {metrics['test_metrics']['mae']:.4f}\n")
            f.write(f"  R²       : {metrics['test_metrics']['r2']:.4f}\n")
        f.write(f"{'='*40}\n")

In [55]:
import numpy as np
import pandas as pd

def prepare_mtl_data_final(df_list, task_names, featurizer):
    # 1. Zebranie wszystkich unikalnych struktur
    all_drugs = set()
    for df in df_list:
        valid = df['Drug'].dropna().astype(str).unique()
        all_drugs.update(valid)

    # 2. Walidacja cząsteczek przez RDKit przed stworzeniem master_list
    # To zapobiega rozbieżnościom rozmiarów X i y
    safe_master_list = []
    for drug in sorted(list(all_drugs)):
        mol = Chem.MolFromSmiles(drug)
        if mol: safe_master_list.append(drug)

    drug_to_idx = {drug: i for i, drug in enumerate(safe_master_list)}
    n_samples = len(safe_master_list)

    # 3. Generowanie X (tylko dla poprawnych cząsteczek)
    df_temp = pd.DataFrame({'Drug': safe_master_list})
    X_features, _ = featurizer(df_temp)

    # Sprawdzenie czy X zawiera NaN (zabezpieczenie przed błędami featurizera)
    if np.isnan(X_features).any():
        X_features = np.nan_to_num(X_features)

    # 4. Mapowanie etykiet y (z zachowaniem NaN tylko w etykietach!)
    y_dict = {}
    for df, task in zip(df_list, task_names):
        y_vec = np.full((n_samples, 1), np.nan, dtype=np.float32)
        # Tworzymy mapę {Drug: Wynik}
        mapping = dict(zip(df['Drug'].astype(str), df['Y']))

        for drug, val in mapping.items():
            if drug in drug_to_idx and not pd.isna(val):
                y_vec[drug_to_idx[drug]] = val
        y_dict[task] = y_vec

    return X_features, y_dict

# Test1: Caco-2 (Wang) + Lipophilicity (Astra Zeneca)

Test pierwszy sprawdzający czy Lipophilicity (42000 związków) pomoże związanemu z nim CaCo-2 (Wang). Sprawdzamy czy MTL da lepsze wyniki niż STL.

In [64]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Caco2_Wang', 'Lipophilicity_AZ']
class_tasks = [] # Dodaj tu zadania klasyfikacyjne
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_absorpcja_fingerprints.txt"

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

# 1. Ładowanie danych
train_caco2, test_caco2 = load_split_pickle('Caco2_Wang')
train_lipo, test_lipo = load_split_pickle('Lipophilicity_AstraZeneca')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_final([train_caco2, train_lipo], reg_tasks, featurizer)
X_test_mtl, y_test_dict = prepare_mtl_data_final([test_caco2, test_lipo], reg_tasks, featurizer)

# Define all loss weighting schemes to test
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKÓW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)

    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadań regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        # Przygotowanie formatu pod Twoje funkcje (opakowanie w 'test_metrics')
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadań klasyfikacyjnych (jeśli istnieją)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypełnienie, jeśli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics( formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostały zapisane w: {filepath}")

Streaming output truncated to the last 5000 lines.
[19:38:45] DEPRECATION WARNING: please use MorganGenerator
[19:38:45] DEPRECATION WARNING: please use MorganGenerator
[19:38:45] DEPRECATION WARNING: please use MorganGenerator
[19:38:45] DEPRECATION WARNING: please use MorganGenerator
[19:38:45] DEPRECATION WARNING: please use MorganGenerator
[19:38:45] DEPRECATION WARNING: please use MorganGenerator
[19:38:45] DEPRECATION WARNING: please use MorganGenerator
[19:38:45] DEPRECATION WARNING: please use MorganGenerator
[19:38:45] DEPRECATION WARNING: please use MorganGenerator
[19:38:45] DEPRECATION WARNING: please use MorganGenerator
[19:38:45] DEPRECATION WARNING: please use MorganGenerator
[19:38:45] DEPRECATION WARNING: please use MorganGenerator
[19:38:45] DEPRECATION WARNING: please use MorganGenerator
[19:38:45] DEPRECATION WARNING: please use MorganGenerator
[19:38:45] DEPRECATION WARNING: please use MorganGenerator
[19:38:45] DEPRECATION WARNING: please use MorganGenerator
[19:3


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 2.1779
  Epoka  20 | Total Loss: 0.4722
  Epoka  40 | Total Loss: 0.1810
  Epoka  60 | Total Loss: 0.0909
  Epoka  80 | Total Loss: 0.0594

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Caco2_Wang

  Loss Weighting: Weighted Loss Sum
  RMSE     : 0.4999
  MAE      : 0.3880
  R²       : 0.6065


Wyniki dla zadania (Regresja): Lipophilicity_AZ

  Loss Weighting: Weighted Loss Sum
  RMSE     : 0.7854
  MAE      : 0.5876
  R²       : 0.5825


--- START TRENINGU (Multi-Task - Uniform Average) ---
  Epoka   0 | Total Loss: 1.1685
  Epoka  20 | Total Loss: 0.1745
  Epoka  40 | Total Loss: 0.0705
  Epoka  60 | Total Loss: 0.0379
  Epoka  80 | Total Loss: 0.0263

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Uniform Average

Wyniki dla zadania (Regresja): Caco2_Wang

  Loss Weighting: Uniform Average
  RMSE     : 0.4961
  MAE      : 0.3823
  R²       : 0.6126


Wyniki dla

# Test2: Caco-2 (Wang) + Lipophilicity (Astra Zeneca) + Solubility (AqSolDB)

Sprawdzamy czy dołożenie kolejnego powiązanego zbioru poprawi wynik.

In [65]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Caco2_Wang', 'Lipophilicity_AZ', 'Solubility_AqSolDB']
class_tasks = [] # Dodaj tu zadania klasyfikacyjne
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_absorpcja_fingerprints.txt"

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

# 1. Ładowanie danych
train_caco2, test_caco2 = load_split_pickle('Caco2_Wang')
train_lipo,  test_lipo  = load_split_pickle('Lipophilicity_AstraZeneca')
train_sol,   test_sol   = load_split_pickle('Solubility_AqSolDB')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_final([train_caco2, train_lipo, train_sol], reg_tasks, featurizer)
X_test_mtl,  y_test_dict  = prepare_mtl_data_final([test_caco2,  test_lipo,  test_sol],  reg_tasks, featurizer)

# Define all loss weighting schemes to test
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKÓW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)

    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadań regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadań klasyfikacyjnych (jeśli istnieją)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypełnienie, jeśli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostały zapisane w: {filepath}")

Streaming output truncated to the last 5000 lines.
[19:39:06] DEPRECATION WARNING: please use MorganGenerator
[19:39:06] DEPRECATION WARNING: please use MorganGenerator
[19:39:06] DEPRECATION WARNING: please use MorganGenerator
[19:39:06] DEPRECATION WARNING: please use MorganGenerator
[19:39:06] DEPRECATION WARNING: please use MorganGenerator
[19:39:06] DEPRECATION WARNING: please use MorganGenerator
[19:39:06] DEPRECATION WARNING: please use MorganGenerator
[19:39:06] DEPRECATION WARNING: please use MorganGenerator
[19:39:06] DEPRECATION WARNING: please use MorganGenerator
[19:39:06] DEPRECATION WARNING: please use MorganGenerator
[19:39:06] DEPRECATION WARNING: please use MorganGenerator
[19:39:06] DEPRECATION WARNING: please use MorganGenerator
[19:39:06] DEPRECATION WARNING: please use MorganGenerator
[19:39:06] DEPRECATION WARNING: please use MorganGenerator
[19:39:06] DEPRECATION WARNING: please use MorganGenerator
[19:39:06] DEPRECATION WARNING: please use MorganGenerator
[19:3


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 3.6804
  Epoka  20 | Total Loss: 0.9108
  Epoka  40 | Total Loss: 0.4204
  Epoka  60 | Total Loss: 0.2337
  Epoka  80 | Total Loss: 0.1596

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Caco2_Wang

  Loss Weighting: Weighted Loss Sum
  RMSE     : 0.5198
  MAE      : 0.3958
  R²       : 0.5746


Wyniki dla zadania (Regresja): Lipophilicity_AZ

  Loss Weighting: Weighted Loss Sum
  RMSE     : 0.7892
  MAE      : 0.5816
  R²       : 0.5785


Wyniki dla zadania (Regresja): Solubility_AqSolDB

  Loss Weighting: Weighted Loss Sum
  RMSE     : 1.2888
  MAE      : 0.9452
  R²       : 0.6939


--- START TRENINGU (Multi-Task - Uniform Average) ---
  Epoka   0 | Total Loss: 1.2774
  Epoka  20 | Total Loss: 0.2847
  Epoka  40 | Total Loss: 0.1276
  Epoka  60 | Total Loss: 0.0736
  Epoka  80 | Total Loss: 0.0517

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Uniform Average



# Test3:  Caco-2 (Wang) + Lipophilicity (Astra Zeneca) + Solubility (AqSolDB) + HIA (Hou)

Sprawdzamy czy dołożenie kolejnego powiązanego zbioru poprawi wynik. Czy w jakimś momencie wyniki przestają się poprawiać?

In [66]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Caco2_Wang', 'Lipophilicity_AZ', 'Solubility_AqSolDB']
class_tasks = ['HIA_Hou'] # Dodaj tu zadania klasyfikacyjne
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_absorpcja_fingerprints.txt"

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

# 1. Ładowanie danych
train_caco2, test_caco2 = load_split_pickle('Caco2_Wang')
train_lipo,  test_lipo  = load_split_pickle('Lipophilicity_AstraZeneca')
train_sol,   test_sol   = load_split_pickle('Solubility_AqSolDB')
train_hia,   test_hia   = load_split_pickle('HIA_Hou')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_final([train_caco2, train_lipo, train_sol, train_hia], reg_tasks + class_tasks, featurizer)
X_test_mtl,  y_test_dict  = prepare_mtl_data_final([test_caco2,  test_lipo,  test_sol,  test_hia],  reg_tasks + class_tasks, featurizer)

# Define all loss weighting schemes to test
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKÓW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)

    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadań regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadań klasyfikacyjnych (jeśli istnieją)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypełnienie, jeśli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostały zapisane w: {filepath}")

Streaming output truncated to the last 5000 lines.
[19:39:29] DEPRECATION WARNING: please use MorganGenerator
[19:39:29] DEPRECATION WARNING: please use MorganGenerator
[19:39:29] DEPRECATION WARNING: please use MorganGenerator
[19:39:29] DEPRECATION WARNING: please use MorganGenerator
[19:39:29] DEPRECATION WARNING: please use MorganGenerator
[19:39:29] DEPRECATION WARNING: please use MorganGenerator
[19:39:29] DEPRECATION WARNING: please use MorganGenerator
[19:39:29] DEPRECATION WARNING: please use MorganGenerator
[19:39:29] DEPRECATION WARNING: please use MorganGenerator
[19:39:29] DEPRECATION WARNING: please use MorganGenerator
[19:39:29] DEPRECATION WARNING: please use MorganGenerator
[19:39:29] DEPRECATION WARNING: please use MorganGenerator
[19:39:29] DEPRECATION WARNING: please use MorganGenerator
[19:39:29] DEPRECATION WARNING: please use MorganGenerator
[19:39:29] DEPRECATION WARNING: please use MorganGenerator
[19:39:29] DEPRECATION WARNING: please use MorganGenerator
[19:3


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 5.2150
  Epoka  20 | Total Loss: 1.2158
  Epoka  40 | Total Loss: 0.5541
  Epoka  60 | Total Loss: 0.2874
  Epoka  80 | Total Loss: 0.1803

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Caco2_Wang

  Loss Weighting: Weighted Loss Sum
  RMSE     : 0.5340
  MAE      : 0.4188
  R²       : 0.5511


Wyniki dla zadania (Regresja): Lipophilicity_AZ

  Loss Weighting: Weighted Loss Sum
  RMSE     : 0.7953
  MAE      : 0.5916
  R²       : 0.5719


Wyniki dla zadania (Regresja): Solubility_AqSolDB

  Loss Weighting: Weighted Loss Sum
  RMSE     : 1.2809
  MAE      : 0.9407
  R²       : 0.6977


Wyniki dla zadania (Klasyfikacja): HIA_Hou

  Loss Weighting: Weighted Loss Sum
  Accuracy : 0.9138
  F1       : 0.9510
  AUROC    : 0.9023


--- START TRENINGU (Multi-Task - Uniform Average) ---
  Epoka   0 | Total Loss: 1.1184
  Epoka  20 | Total Loss: 0.2770
  Epoka  40 | Total Loss: 0

# Test4: Caco-2 (Wang) + HIA (Hou)

Sprawdzamy czy mały zbiór HIA pomoże CaCo-2 mniej niż większe zbiory z poprzednich testów. Jednocześnie sparwdzamy STL vs MTL.

In [67]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Caco2_Wang']
class_tasks = ['HIA_Hou'] # Dodaj tu zadania klasyfikacyjne
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_absorpcja_fingerprints.txt"

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

# 1. Ładowanie danych
train_caco2, test_caco2 = load_split_pickle('Caco2_Wang')
train_hia,   test_hia   = load_split_pickle('HIA_Hou')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_final([train_caco2, train_hia], reg_tasks + class_tasks, featurizer)
X_test_mtl,  y_test_dict  = prepare_mtl_data_final([test_caco2,  test_hia],  reg_tasks + class_tasks, featurizer)

# Define all loss weighting schemes to test
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKÓW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)

    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadań regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadań klasyfikacyjnych (jeśli istnieją)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypełnienie, jeśli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostały zapisane w: {filepath}")

[19:39:42] DEPRECATION WARNING: please use MorganGenerator
[19:39:42] DEPRECATION WARNING: please use MorganGenerator
[19:39:42] DEPRECATION WARNING: please use MorganGenerator
[19:39:42] DEPRECATION WARNING: please use MorganGenerator
[19:39:42] DEPRECATION WARNING: please use MorganGenerator
[19:39:42] DEPRECATION WARNING: please use MorganGenerator
[19:39:42] DEPRECATION WARNING: please use MorganGenerator
[19:39:42] DEPRECATION WARNING: please use MorganGenerator
[19:39:42] DEPRECATION WARNING: please use MorganGenerator
[19:39:42] DEPRECATION WARNING: please use MorganGenerator
[19:39:42] DEPRECATION WARNING: please use MorganGenerator
[19:39:42] DEPRECATION WARNING: please use MorganGenerator
[19:39:42] DEPRECATION WARNING: please use MorganGenerator
[19:39:42] DEPRECATION WARNING: please use MorganGenerator
[19:39:42] DEPRECATION WARNING: please use MorganGenerator
[19:39:42] DEPRECATION WARNING: please use MorganGenerator
[19:39:42] DEPRECATION WARNING: please use MorganGenerat


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 1.9319
  Epoka  20 | Total Loss: 0.3423
  Epoka  40 | Total Loss: 0.0701
  Epoka  60 | Total Loss: 0.0339
  Epoka  80 | Total Loss: 0.0250

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Caco2_Wang

  Loss Weighting: Weighted Loss Sum
  RMSE     : 0.5069
  MAE      : 0.3942
  R²       : 0.5955


Wyniki dla zadania (Klasyfikacja): HIA_Hou

  Loss Weighting: Weighted Loss Sum
  Accuracy : 0.9138
  F1       : 0.9510
  AUROC    : 0.9327


--- START TRENINGU (Multi-Task - Uniform Average) ---
  Epoka   0 | Total Loss: 1.0021
  Epoka  20 | Total Loss: 0.1519
  Epoka  40 | Total Loss: 0.0500
  Epoka  60 | Total Loss: 0.0191
  Epoka  80 | Total Loss: 0.0134

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Uniform Average

Wyniki dla zadania (Regresja): Caco2_Wang

  Loss Weighting: Uniform Average
  RMSE     : 0.4949
  MAE      : 0.3850
  R²       : 0.6144


Wyniki dla zada

# Test5: Lipophilicity (Astra Zeneca) + Solubility (AqSolDB)

Sprawdzamy czy duży zbiór Solubility pomoże innemu dużemu zbiorowi mniej niż małemu CaCo-2. Jednocześnie sprawdzamy STL vs MTL.

In [68]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Lipophilicity_AZ', 'Solubility_AqSolDB']
class_tasks = [] # Dodaj tu zadania klasyfikacyjne
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_absorpcja_fingerprints.txt"

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

# 1. Ładowanie danych
train_lipo, test_lipo = load_split_pickle('Lipophilicity_AstraZeneca')
train_sol,  test_sol  = load_split_pickle('Solubility_AqSolDB')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_final([train_lipo, train_sol], reg_tasks, featurizer)
X_test_mtl,  y_test_dict  = prepare_mtl_data_final([test_lipo,  test_sol],  reg_tasks, featurizer)

# Define all loss weighting schemes to test
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKÓW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)

    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadań regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadań klasyfikacyjnych (jeśli istnieją)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypełnienie, jeśli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostały zapisane w: {filepath}")

Streaming output truncated to the last 5000 lines.
[19:39:57] DEPRECATION WARNING: please use MorganGenerator
[19:39:57] DEPRECATION WARNING: please use MorganGenerator
[19:39:57] DEPRECATION WARNING: please use MorganGenerator
[19:39:57] DEPRECATION WARNING: please use MorganGenerator
[19:39:57] DEPRECATION WARNING: please use MorganGenerator
[19:39:57] DEPRECATION WARNING: please use MorganGenerator
[19:39:57] DEPRECATION WARNING: please use MorganGenerator
[19:39:57] DEPRECATION WARNING: please use MorganGenerator
[19:39:57] DEPRECATION WARNING: please use MorganGenerator
[19:39:57] DEPRECATION WARNING: please use MorganGenerator
[19:39:57] DEPRECATION WARNING: please use MorganGenerator
[19:39:57] DEPRECATION WARNING: please use MorganGenerator
[19:39:57] DEPRECATION WARNING: please use MorganGenerator
[19:39:57] DEPRECATION WARNING: please use MorganGenerator
[19:39:57] DEPRECATION WARNING: please use MorganGenerator
[19:39:57] DEPRECATION WARNING: please use MorganGenerator
[19:3


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 2.4853
  Epoka  20 | Total Loss: 1.2005
  Epoka  40 | Total Loss: 0.6252
  Epoka  60 | Total Loss: 0.3215
  Epoka  80 | Total Loss: 0.1711

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Lipophilicity_AZ

  Loss Weighting: Weighted Loss Sum
  RMSE     : 0.8060
  MAE      : 0.5944
  R²       : 0.5603


Wyniki dla zadania (Regresja): Solubility_AqSolDB

  Loss Weighting: Weighted Loss Sum
  RMSE     : 1.2862
  MAE      : 0.9468
  R²       : 0.6951


--- START TRENINGU (Multi-Task - Uniform Average) ---
  Epoka   0 | Total Loss: 1.1319
  Epoka  20 | Total Loss: 0.2763
  Epoka  40 | Total Loss: 0.1283
  Epoka  60 | Total Loss: 0.0709
  Epoka  80 | Total Loss: 0.0474

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Uniform Average

Wyniki dla zadania (Regresja): Lipophilicity_AZ

  Loss Weighting: Uniform Average
  RMSE     : 0.8079
  MAE      : 0.5924
  R²       : 0.558

# Test6: Caco-2 (Wang) + AMES Mutagenicity

Sprawdzamy czy zbiór niepowiązany pomoże zbiorowi CaCo-2.

In [69]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Caco2_Wang']
class_tasks = ['AMES'] # Dodaj tu zadania klasyfikacyjne
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_absorpcja_fingerprints.txt"

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

# 1. Ładowanie danych
train_caco2, test_caco2 = load_split_pickle('Caco2_Wang')
train_ames,  test_ames  = load_split_pickle('AMES')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_final([train_caco2, train_ames], reg_tasks + class_tasks, featurizer)
X_test_mtl,  y_test_dict  = prepare_mtl_data_final([test_caco2,  test_ames],  reg_tasks + class_tasks, featurizer)

# Define all loss weighting schemes to test
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKÓW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)

    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadań regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadań klasyfikacyjnych (jeśli istnieją)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypełnienie, jeśli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostały zapisane w: {filepath}")

Streaming output truncated to the last 5000 lines.
[19:40:12] DEPRECATION WARNING: please use MorganGenerator
[19:40:12] DEPRECATION WARNING: please use MorganGenerator
[19:40:12] DEPRECATION WARNING: please use MorganGenerator
[19:40:12] DEPRECATION WARNING: please use MorganGenerator
[19:40:12] DEPRECATION WARNING: please use MorganGenerator
[19:40:12] DEPRECATION WARNING: please use MorganGenerator
[19:40:12] DEPRECATION WARNING: please use MorganGenerator
[19:40:12] DEPRECATION WARNING: please use MorganGenerator
[19:40:12] DEPRECATION WARNING: please use MorganGenerator
[19:40:12] DEPRECATION WARNING: please use MorganGenerator
[19:40:12] DEPRECATION WARNING: please use MorganGenerator
[19:40:12] DEPRECATION WARNING: please use MorganGenerator
[19:40:12] DEPRECATION WARNING: please use MorganGenerator
[19:40:12] DEPRECATION WARNING: please use MorganGenerator
[19:40:12] DEPRECATION WARNING: please use MorganGenerator
[19:40:12] DEPRECATION WARNING: please use MorganGenerator
[19:4


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 2.0304
  Epoka  20 | Total Loss: 0.8895
  Epoka  40 | Total Loss: 0.4542
  Epoka  60 | Total Loss: 0.2874
  Epoka  80 | Total Loss: 0.1377

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Caco2_Wang

  Loss Weighting: Weighted Loss Sum
  RMSE     : 0.4985
  MAE      : 0.3880
  R²       : 0.6088


Wyniki dla zadania (Klasyfikacja): AMES

  Loss Weighting: Weighted Loss Sum
  Accuracy : 0.8191
  F1       : 0.8343
  AUROC    : 0.8862


--- START TRENINGU (Multi-Task - Uniform Average) ---
  Epoka   0 | Total Loss: 0.9489
  Epoka  20 | Total Loss: 0.3330
  Epoka  40 | Total Loss: 0.2218
  Epoka  60 | Total Loss: 0.1440
  Epoka  80 | Total Loss: 0.0771

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Uniform Average

Wyniki dla zadania (Regresja): Caco2_Wang

  Loss Weighting: Uniform Average
  RMSE     : 0.5063
  MAE      : 0.3914
  R²       : 0.5964


Wyniki dla zadania

# Test7: Caco-2 (Wang)+ Lipophilicity (Astra Zeneca)  + AMES Mutagenicity

In [70]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Caco2_Wang', 'Lipophilicity_AZ']
class_tasks = ['AMES'] # Dodaj tu zadania klasyfikacyjne
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_absorpcja_fingerprints.txt"

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

# 1. Ladowanie danych
train_caco2, test_caco2 = load_split_pickle('Caco2_Wang')
train_lipo, test_lipo = load_split_pickle('Lipophilicity_AstraZeneca')
train_ames, test_ames = load_split_pickle('AMES')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_final([train_caco2, train_lipo, train_ames], reg_tasks + class_tasks, featurizer)
X_test_mtl,  y_test_dict  = prepare_mtl_data_final([test_caco2, test_lipo, test_ames], reg_tasks + class_tasks, featurizer)

# Define all loss weighting schemes to test
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKOW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)

    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadan regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadan klasyfikacyjnych (jesli istnieja)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypelnienie, jesli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostaly zapisane w: {filepath}")

Streaming output truncated to the last 5000 lines.
[19:40:31] DEPRECATION WARNING: please use MorganGenerator
[19:40:31] DEPRECATION WARNING: please use MorganGenerator
[19:40:31] DEPRECATION WARNING: please use MorganGenerator
[19:40:31] DEPRECATION WARNING: please use MorganGenerator
[19:40:31] DEPRECATION WARNING: please use MorganGenerator
[19:40:31] DEPRECATION WARNING: please use MorganGenerator
[19:40:31] DEPRECATION WARNING: please use MorganGenerator
[19:40:31] DEPRECATION WARNING: please use MorganGenerator
[19:40:31] DEPRECATION WARNING: please use MorganGenerator
[19:40:31] DEPRECATION WARNING: please use MorganGenerator
[19:40:31] DEPRECATION WARNING: please use MorganGenerator
[19:40:31] DEPRECATION WARNING: please use MorganGenerator
[19:40:31] DEPRECATION WARNING: please use MorganGenerator
[19:40:31] DEPRECATION WARNING: please use MorganGenerator
[19:40:31] DEPRECATION WARNING: please use MorganGenerator
[19:40:31] DEPRECATION WARNING: please use MorganGenerator
[19:4


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 3.3508
  Epoka  20 | Total Loss: 1.2987
  Epoka  40 | Total Loss: 0.7223
  Epoka  60 | Total Loss: 0.4263
  Epoka  80 | Total Loss: 0.2215

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Caco2_Wang

  Loss Weighting: Weighted Loss Sum
  RMSE     : 0.5335
  MAE      : 0.3900
  R²       : 0.5520


Wyniki dla zadania (Regresja): Lipophilicity_AZ

  Loss Weighting: Weighted Loss Sum
  RMSE     : 0.8203
  MAE      : 0.6073
  R²       : 0.5446


Wyniki dla zadania (Klasyfikacja): AMES

  Loss Weighting: Weighted Loss Sum
  Accuracy : 0.8226
  F1       : 0.8383
  AUROC    : 0.8922


--- START TRENINGU (Multi-Task - Uniform Average) ---
  Epoka   0 | Total Loss: 1.0231
  Epoka  20 | Total Loss: 0.4125
  Epoka  40 | Total Loss: 0.2459
  Epoka  60 | Total Loss: 0.1490
  Epoka  80 | Total Loss: 0.0923

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Uniform Average

Wyniki dla

#Test 8: Caco-2 (Wang) + Lipophilicity (Astra Zeneca) + Solubility (AqSolDB) + HIA (Hou)  + AMES

In [71]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Caco2_Wang', 'Lipophilicity_AZ', 'Solubility_AqSolDB']
class_tasks = ['HIA_Hou', 'AMES'] # Dodaj tu zadania klasyfikacyjne
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_absorpcja_fingerprints.txt"

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

# 1. Ladowanie danych
train_caco2, test_caco2 = load_split_pickle('Caco2_Wang')
train_lipo, test_lipo = load_split_pickle('Lipophilicity_AstraZeneca')
train_sol, test_sol = load_split_pickle('Solubility_AqSolDB')
train_hia, test_hia = load_split_pickle('HIA_Hou')
train_ames, test_ames = load_split_pickle('AMES')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_final([train_caco2, train_lipo, train_sol, train_hia, train_ames], reg_tasks + class_tasks, featurizer)
X_test_mtl,  y_test_dict  = prepare_mtl_data_final([test_caco2, test_lipo, test_sol, test_hia, test_ames], reg_tasks + class_tasks, featurizer)

# Define all loss weighting schemes to test
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKOW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)

    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadan regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadan klasyfikacyjnych (jesli istnieja)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypelnienie, jesli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostaly zapisane w: {filepath}")

Streaming output truncated to the last 5000 lines.
[19:41:01] DEPRECATION WARNING: please use MorganGenerator
[19:41:01] DEPRECATION WARNING: please use MorganGenerator
[19:41:01] DEPRECATION WARNING: please use MorganGenerator
[19:41:01] WARNING: not removing hydrogen atom without neighbors
[19:41:01] WARNING: not removing hydrogen atom without neighbors
[19:41:01] DEPRECATION WARNING: please use MorganGenerator
[19:41:01] DEPRECATION WARNING: please use MorganGenerator
[19:41:01] DEPRECATION WARNING: please use MorganGenerator
[19:41:01] DEPRECATION WARNING: please use MorganGenerator
[19:41:01] DEPRECATION WARNING: please use MorganGenerator
[19:41:01] DEPRECATION WARNING: please use MorganGenerator
[19:41:01] DEPRECATION WARNING: please use MorganGenerator
[19:41:01] DEPRECATION WARNING: please use MorganGenerator
[19:41:01] DEPRECATION WARNING: please use MorganGenerator
[19:41:01] DEPRECATION WARNING: please use MorganGenerator
[19:41:01] DEPRECATION WARNING: please use MorganGen


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 4.9670
  Epoka  20 | Total Loss: 1.5848
  Epoka  40 | Total Loss: 0.8131
  Epoka  60 | Total Loss: 0.4512
  Epoka  80 | Total Loss: 0.2540

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Caco2_Wang

  Loss Weighting: Weighted Loss Sum
  RMSE     : 0.5187
  MAE      : 0.3915
  R²       : 0.5765


Wyniki dla zadania (Regresja): Lipophilicity_AZ

  Loss Weighting: Weighted Loss Sum
  RMSE     : 0.7961
  MAE      : 0.5836
  R²       : 0.5711


Wyniki dla zadania (Regresja): Solubility_AqSolDB

  Loss Weighting: Weighted Loss Sum
  RMSE     : 1.2961
  MAE      : 0.9575
  R²       : 0.6904


Wyniki dla zadania (Klasyfikacja): HIA_Hou

  Loss Weighting: Weighted Loss Sum
  Accuracy : 0.9052
  F1       : 0.9463
  AUROC    : 0.8991


Wyniki dla zadania (Klasyfikacja): AMES

  Loss Weighting: Weighted Loss Sum
  Accuracy : 0.8246
  F1       : 0.8383
  AUROC    : 0.8910


--- STAR